In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")
print(clf("I love this project idea!"))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998812675476074}]


In [ ]:
import pandas as pd

# 1. Load both CSVs
df_media = pd.read_csv("media_dataset.csv")
df_train2 = pd.read_csv("training_data.csv")

print("media_dataset shape:", df_media.shape)
print("training_data shape:", df_train2.shape)

print("\nmedia_dataset columns:", df_media.columns.tolist())
print("training_data-2 columns:", df_train2.columns.tolist())

print("\nSample from media_dataset:")
display(df_media.head(5))

print("\nSample from training_data:")
display(df_train2.head(5))

print("\nGenre counts in media_dataset:")
print(df_media["genre"].value_counts())

print("\nGenre counts in training_data:")
print(df_train2["genre"].value_counts())


media_dataset shape: (700, 2)
training_data shape: (700, 2)

media_dataset columns: ['synopsis', 'genre']
training_data-2 columns: ['synopsis', 'genre']

Sample from media_dataset:


,synopsis,genre
0,A genius high school student finds a supernatu...,Psychological
1,A young pirate sets sail to find the legendary...,Action
2,A reserved high school boy discovers that his ...,Romance
3,A high school volleyball team fights its way f...,Other
4,A lonely slime reincarnated from a dead salary...,Fantasy



Sample from training_data:


,synopsis,genre
0,Two high school students discover that their o...,Romance
1,The top two students of an elite academy's stu...,Romance
2,A gentle but intimidating-looking high school ...,Romance
3,A former child prodigy pianist loses his abili...,Romance
4,After a family tragedy turns her life upside d...,Romance



Genre counts in media_dataset:
genre
Psychological    100
Action           100
Romance          100
Other            100
Fantasy          100
Thriller         100
Slice-of-life    100
Name: count, dtype: int64

Genre counts in training_data:
genre
Fantasy          109
Thriller         107
Romance           99
Psychological     99
Slice-of-life     98
Action            98
Other             90
Name: count, dtype: int64


In [ ]:

df_media = df_media.rename(columns={"synopsis": "text"})
df_train2 = df_train2.rename(columns={"synopsis": "text"})

df_media = df_media[["text", "genre"]]
df_train2 = df_train2[["text", "genre"]]

df_all = pd.concat([df_media, df_train2], ignore_index=True)

df_all = df_all.drop_duplicates()

print("Merged dataset shape:", df_all.shape)
print("\nGenre counts in merged data:")
print(df_all["genre"].value_counts())


Merged dataset shape: (1400, 2)

Genre counts in merged data:
genre
Fantasy          209
Thriller         207
Psychological    199
Romance          199
Action           198
Slice-of-life    198
Other            190
Name: count, dtype: int64


In [ ]:
df_all.to_csv("genre_data_merged.csv", index=False)
print("Saved merged dataset as genre_data_merged.csv")


Saved merged dataset as genre_data_merged.csv


In [1]:
from datasets import load_dataset

# Load the CSV as a Hugging Face DatasetDict
raw_datasets = load_dataset(
    "csv",
    data_files={"train": "genre_data_merged.csv"},
)

raw_datasets


Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'genre'],
        num_rows: 1400
    })
})

In [2]:
# 1. Define the 7 genre names in a fixed order
label_names = [
    "Romance",
    "Slice-of-life",
    "Action",
    "Fantasy",
    "Psychological",
    "Thriller",
    "Other",
]

# 2. Simple cleaning: strip whitespace from genre strings
def clean_genre(example):
    example["genre"] = example["genre"].strip()
    return example

datasets_clean = raw_datasets.map(clean_genre)

# 3. Show the unique genres present (sanity check)
unique_genres = set(datasets_clean["train"]["genre"])
print("Unique genres in data:", unique_genres)


Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

Unique genres in data: {'Fantasy', 'Romance', 'Action', 'Thriller', 'Psychological', 'Other', 'Slice-of-life'}


In [3]:
# 1. Map label name -> id, and id -> name
label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for name, i in label2id.items()}

print("label2id:", label2id)

# 2. Add a new 'label' column with integer ids
def add_label(example):
    example["label"] = label2id[example["genre"]]
    return example

datasets_labeled = datasets_clean.map(add_label)

# 3. Inspect one row to confirm
print(datasets_labeled["train"][0])


label2id: {'Romance': 0, 'Slice-of-life': 1, 'Action': 2, 'Fantasy': 3, 'Psychological': 4, 'Thriller': 5, 'Other': 6}


Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

{'text': 'A genius high school student finds a supernatural notebook that kills anyone whose name is written in it. He launches a secret crusade against criminals, drawing the attention of an equally brilliant detective.', 'genre': 'Psychological', 'label': 4}


In [4]:
datasets_labeled = datasets_labeled.class_encode_column("genre")
# Start from the single 'train' split
full_train = datasets_labeled["train"]

# First split: 80% train, 20% temp (val+test)
train_testvalid = full_train.train_test_split(
    test_size=0.2,
    stratify_by_column="genre",  #- keep genre distribution balanced
)

# Second split: split temp into validation and test (50/50 of 20% => 10% each)
test_valid = train_testvalid["test"].train_test_split(
    test_size=0.5,
    stratify_by_column="genre",
)

# Build final DatasetDict
datasets = {
    "train": train_testvalid["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"],
}

for split_name, ds in datasets.items():
    print(split_name, "=>", len(ds), "rows")


Casting to class labels:   0%|          | 0/1400 [00:00<?, ? examples/s]

train => 1120 rows
validation => 140 rows
test => 140 rows


In [7]:
from datasets import DatasetDict

# Wrap our splits into a proper DatasetDict (optional but convenient)
datasets = DatasetDict(datasets)

# Inspect label info from the encoded 'genre' column
genre_feature = datasets["train"].features["genre"]
label_names = list(genre_feature.names)

print("Label names:", label_names)

label2id = {name: i for i, name in enumerate(label_names)}
id2label = {i: name for i, name in enumerate(label_names)}
print("label2id:", label2id)


Label names: ['Action', 'Fantasy', 'Other', 'Psychological', 'Romance', 'Slice-of-life', 'Thriller']
label2id: {'Action': 0, 'Fantasy': 1, 'Other': 2, 'Psychological': 3, 'Romance': 4, 'Slice-of-life': 5, 'Thriller': 6}


In [8]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
    )

tokenized_datasets = datasets.map(tokenize_function, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1120 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

Map:   0%|          | 0/140 [00:00<?, ? examples/s]

In [9]:
# For each split: rename 'genre' -> 'labels', drop raw text, set format
for split in ["train", "validation", "test"]:
    ds = tokenized_datasets[split]
    ds = ds.rename_column("genre", "labels")
    ds = ds.remove_columns(["text"])  # keep token ids + labels only
    ds.set_format("torch")
    tokenized_datasets[split] = ds

print(tokenized_datasets["train"][0])


{'labels': tensor(6), 'label': tensor(5), 'input_ids': tensor([  101,  1037,  2177,  1997,  2111,  2006,  1037,  1005,  7484,  1011,
         4507,  1005,  3439,  8592,  5382,  2027,  2024,  2941,  2108,  2109,
         2004,  1005,  8247,  1011,  3231,  2545,  1005,  2005,  1037,  2047,
         1005,  2162,  1011, 12504,  1005,  1998,  1996,  6677,  2024,  4568,
         1012,  2027,  2442,  2224,  2037,  3716,  1997,  1005,  3987,  1011,
         9887,  1005,  2000,  5788,  1996, 23599,  2645,  1998,  2424,  1996,
         1005,  4748, 10020,  1011, 10122,  1012,  1005,  1996,  7984,  2003,
         1037,  9049,  1998, 19118, 13181, 20200,  2645,  1012,   102,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
         

In [10]:
from transformers import AutoModelForSequenceClassification

num_labels = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average="macro")
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
    }


In [16]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="genre_classifier_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)


In [17]:
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.422759,0.892658,0.807143,0.796766
2,0.581925,0.612754,0.814286,0.813424
3,0.306049,0.473930,0.857143,0.850212


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=420, training_loss=0.8319176764715286, metrics={'train_runtime': 105.909, 'train_samples_per_second': 31.725, 'train_steps_per_second': 3.966, 'total_flos': 222565073633280.0, 'train_loss': 0.8319176764715286, 'epoch': 3.0})

In [18]:
trainer.evaluate(tokenized_datasets["test"])


{'eval_loss': 0.4926792085170746,
 'eval_accuracy': 0.8642857142857143,
 'eval_f1_macro': 0.8633558595148639,
 'eval_runtime': 1.2544,
 'eval_samples_per_second': 111.605,
 'eval_steps_per_second': 14.349,
 'epoch': 3.0}

In [19]:
import torch

# Make sure we have the same id2label mapping as before
print(id2label)

def predict_genre(text: str):
    model.eval()
    # Tokenize the input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256,
    ).to(model.device)

    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # Convert to probabilities
    probs = torch.softmax(logits, dim=-1)[0]
    pred_id = int(torch.argmax(probs))
    pred_label = id2label[pred_id]
    pred_conf = float(probs[pred_id])

    return pred_label, pred_conf


{0: 'Action', 1: 'Fantasy', 2: 'Other', 3: 'Psychological', 4: 'Romance', 5: 'Slice-of-life', 6: 'Thriller'}


In [26]:
text = "A shy girl transfers to a new school and slowly falls in love with her club senior."
label, conf = predict_genre(text)
print(label, conf)


Action 0.8243504166603088


In [27]:
text = "A detective hunts a serial killer who leaves puzzles at each crime scene."
label, conf = predict_genre(text)
print(label, conf)


Slice-of-life 0.7944258451461792
